# AuraSight — YOLOv8 Acne Detection Training

**目的：** 用 Roboflow 导出的痘痘数据集在 Google Colab 上训练 YOLOv8 模型，之后把权重上传到后端替换 Roboflow Hosted API。

**流程：**
1. 安装依赖
2. 从 Roboflow 下载数据集
3. 训练 YOLOv8
4. 评估结果（mAP、混淆矩阵）
5. 导出 `best.pt` 权重
6. 下载权重到本地 / 上传到服务器

> ⚠️ 记得在 Colab 顶部菜单 **Runtime → Change runtime type → GPU (T4)** 再跑！

## 0. 检查 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  没有 GPU！请先切换到 GPU runtime 再继续。')

## 1. 安装依赖

In [ ]:
!pip install ultralytics roboflow --quiet
from ultralytics import YOLO
import roboflow
print('ultralytics version:', __import__('ultralytics').__version__)

## 2. 从 Roboflow 下载数据集

把下面的 `RF_API_KEY` 换成你的 Roboflow API Key（在 Roboflow → Settings → API Keys 里找）。

Workspace / Project / Version 也要对上你的项目。

In [ ]:
RF_API_KEY   = "ApbnDOP6wMTly0rp8Mmi"   # ← 你的 Roboflow API Key
RF_WORKSPACE = "your-workspace"          # ← Roboflow workspace slug（URL里的那个）
RF_PROJECT   = "acne-detection-v1"       # ← 项目名 slug
RF_VERSION   = 1                         # ← 版本号

from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print("数据集路径:", dataset.location)
print("data.yaml:", DATA_YAML)

In [ ]:
# 确认 data.yaml 内容
with open(DATA_YAML) as f:
    print(f.read())

## 3. 训练 YOLOv8

| 参数 | 说明 | 建议值 |
|------|------|--------|
| `model` | 基础模型大小 | `yolov8n.pt`（快）/ `yolov8s.pt`（更准） |
| `epochs` | 训练轮数 | 50–100 |
| `imgsz` | 图片尺寸 | 640 |
| `batch` | batch size | -1（自动）或 16 |
| `patience` | 早停轮数 | 20 |

> 1419 张图 + 50 epochs + yolov8n 在 T4 上大约 15–25 分钟。

In [ ]:
model = YOLO("yolov8s.pt")  # yolov8n.pt 更快，yolov8s.pt 更准

results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=-1,          # 自动选 batch size
    patience=20,       # 20轮没有改善就提前停止
    device=0,          # GPU 0
    project="aurasight_runs",
    name="acne_yolov8s",
    exist_ok=True,
    # 数据增强（针对皮肤照片优化）
    hsv_h=0.015,       # 色调轻微变化（皮肤颜色敏感）
    hsv_s=0.5,         # 饱和度变化
    hsv_v=0.4,         # 亮度变化（模拟不同光线）
    fliplr=0.5,        # 左右翻转
    flipud=0.0,        # 不上下翻转（人脸方向重要）
    degrees=10,        # 轻微旋转
    translate=0.1,
    scale=0.3,
    mosaic=0.5,        # Mosaic 增强（适度）
)

print("\n✅ 训练完成！")

## 4. 评估结果

In [ ]:
import os
from IPython.display import Image, display

run_dir = "aurasight_runs/acne_yolov8s"

# 训练曲线
results_png = os.path.join(run_dir, "results.png")
if os.path.exists(results_png):
    display(Image(results_png))

# 混淆矩阵
conf_matrix = os.path.join(run_dir, "confusion_matrix.png")
if os.path.exists(conf_matrix):
    display(Image(conf_matrix))

In [ ]:
# 在验证集上跑 metrics
best_weights = os.path.join(run_dir, "weights", "best.pt")
val_model = YOLO(best_weights)
metrics = val_model.val(data=DATA_YAML)

print(f"mAP@0.5:     {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95:{metrics.box.map:.3f}")
print(f"Precision:   {metrics.box.p[0]:.3f}")
print(f"Recall:      {metrics.box.r[0]:.3f}")

In [ ]:
# 可视化预测（在验证集里随机抽几张）
import glob, random

val_images = glob.glob(dataset.location + "/valid/images/*.jpg")
sample_imgs = random.sample(val_images, min(4, len(val_images)))

pred_results = val_model.predict(source=sample_imgs, conf=0.35, save=True,
                                  project="aurasight_runs", name="preview", exist_ok=True)

for img_path in glob.glob("aurasight_runs/preview/*.jpg"):
    display(Image(img_path, width=400))

## 5. 导出权重

训练完最好的权重在 `aurasight_runs/acne_yolov8s/weights/best.pt`。

可以直接下载，也可以导出成 ONNX 格式（适合服务器部署）。

In [ ]:
# 方式 A：直接下载 .pt 权重（在本地用 Python 推理）
from google.colab import files
files.download(best_weights)
print("✅ best.pt 已下载到本地")

In [ ]:
# 方式 B：导出成 ONNX（服务器上用 onnxruntime 推理，不需要 GPU）
export_model = YOLO(best_weights)
export_model.export(format="onnx", imgsz=640, dynamic=True, simplify=True)

onnx_path = best_weights.replace(".pt", ".onnx")
print("ONNX 路径:", onnx_path)
files.download(onnx_path)

## 6. （可选）把权重上传回 Roboflow

如果你想继续用 Roboflow Hosted API 而不是自己部署，可以把训练好的模型上传到 Roboflow 创建新 version。

In [ ]:
# 把 best.pt 上传到 Roboflow（会创建一个新的 version）
project.version(RF_VERSION).deploy(
    model_type="yolov8",
    model_path=run_dir + "/"
)
print("✅ 模型已上传到 Roboflow，可以在 Hosted API 直接使用新 version")

## 7. 后端集成说明

训练完之后你有两种选择：

### 选项 A — 继续用 Roboflow Hosted API（最省事）
把上传后的新 version 号更新到 `.env`：
```
ROBOFLOW_MODEL=acne-detection-v1-kf14t/2   # 版本号+1
```
后端代码不用改。

### 选项 B — 自己在 Node.js 后端用 ONNX 推理（不耗 Roboflow credits）
下载 `best.onnx`，放到后端项目里，然后用 `onnxruntime-node` 加载：
```bash
npm install onnxruntime-node
```
推理代码我可以帮你写，告诉我你想用哪种方式。

### 选项 C — 上传到 Render / Railway（免费服务器 + Python 推理）
用 FastAPI + ultralytics 包装成一个小 API，部署后替换 `ROBOFLOW_MODEL` 对应的 URL 就行。

---
## 快速参考：Workspace / Project Slug 怎么找

打开你的 Roboflow 项目，URL 格式是：
```
https://app.roboflow.com/<workspace>/<project>/<version>
```
例如：
```
https://app.roboflow.com/kevin-abc123/acne-detection-v1/1
                          ↑workspace      ↑project        ↑version
```
把 `RF_WORKSPACE` 和 `RF_PROJECT` 换成你 URL 里的值就好。